In [1]:
#!/usr/bin/env python3
# =========================================================
# ENSEMBLE SUMMARIZATION SYSTEM WITH RAG
# Two Fine-tuned Models: LED + Legal-Pegasus
# With InLegalBERT MMR Sentence Selection + RAG Retrieval
# =========================================================

import os
import re
import json
import torch
import numpy as np
from tqdm import tqdm
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    LEDTokenizer,
    LEDForConditionalGeneration,
    PegasusTokenizer,
    PegasusForConditionalGeneration,
)
from sentence_transformers import SentenceTransformer, util
from collections import Counter
import evaluate
import nltk

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Device: {device}")

# =========================================================
# CONFIGURATION
# =========================================================

# Model Paths / Names
MODEL_PATHS = {
    "LED":     "LED",                      # local fine-tuned LED checkpoint
    "PEGASUS": "nsi319/legal-pegasus"      # Legal-Pegasus (HuggingFace hub or local)
}

# Data paths
TEST_JSON_PATH  = "test.json"
OUTPUT_DIR      = "./ensemble_rag_results"

# Model-specific generation parameters
MODEL_CONFIG = {
    "LED": {
        "max_input":  4096,   # LED handles long inputs natively
        "max_output":  512,
        "num_beams":     4
    },
    "PEGASUS": {
        "max_input":  1024,
        "max_output":  256,
        "num_beams":     4
    }
}

# ── RAG parameters (from Script 1) ──────────────────────
RETRIEVER_MODEL       = "law-ai/InLegalBERT"   # dense retriever
CHUNK_SIZE_TOKENS     = 800
CHUNK_OVERLAP_TOKENS  = 200
TOP_K                 = 8        # initial chunks to retrieve
MAX_RETRIEVE_ROUNDS   = 3        # expansion rounds if summary too short
RETRIEVE_BY           = "token"  # "sentence" or "token"

# Word-length targets for final summaries
MIN_WORDS = 400
MAX_WORDS = 500
TOK_PER_WORD    = 1.5
GEN_MIN_TOKENS  = int(MIN_WORDS * TOK_PER_WORD)   # ≈ 600
GEN_MAX_TOKENS  = int(MAX_WORDS * TOK_PER_WORD)   # ≈ 750

# ── MMR parameters ───────────────────────────────────────
MMR_K        = 60    # for PEGASUS (shorter context)
MMR_K_LED    = 200   # for LED (longer context)
MMR_LAMBDA   = 0.7

# ── Ensemble weights ─────────────────────────────────────
ENSEMBLE_WEIGHTS = {
    "LED":     0.50,
    "PEGASUS": 0.50
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📂 Output directory: {OUTPUT_DIR}")
print(f"📂 Test data:        {TEST_JSON_PATH}")

# =========================================================
# UTILITIES  (from Script 1)
# =========================================================

def clean_judgment_text(text):
    text = re.sub(r"\[Page No\.\s*\d+\]", " ", text)
    text = re.sub(r"Case\s*:-.*?\n", " ", text)
    text = re.sub(r"\(\d+\)", "", text)
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    text = text.replace(" ,", ",").replace(" .", ".").strip()
    return text

def words_count(text):
    return len(re.findall(r"\w+", text))

def truncate_to_word_limit_by_sentences(summary, min_words=MIN_WORDS, max_words=MAX_WORDS):
    words = summary.split()
    if min_words <= len(words) <= max_words:
        return summary
    sents = sent_tokenize(summary)
    out, w = [], 0
    for s in sents:
        sw = len(s.split())
        if w + sw > max_words:
            break
        out.append(s)
        w += sw
    if w < min_words:
        for s in sents[len(out):]:
            out.append(s)
            w += len(s.split())
            if w >= min_words:
                break
    return " ".join(out)

def chunk_by_tokens(text, tokenizer, max_tokens=CHUNK_SIZE_TOKENS, overlap=CHUNK_OVERLAP_TOKENS):
    """Token-based chunking with overlap (Script 1 approach)."""
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) <= max_tokens:
        return [tokenizer.decode(tokens, clean_up_tokenization_spaces=True)]
    chunks, step = [], max_tokens - overlap
    for i in range(0, len(tokens), step):
        chunk_tokens = tokens[i:i + max_tokens]
        if not chunk_tokens:
            break
        chunks.append(tokenizer.decode(chunk_tokens, clean_up_tokenization_spaces=True))
        if i + max_tokens >= len(tokens):
            break
    return chunks

def chunk_into_sentences(text, n=8):
    """Sentence-group chunking (Script 1 approach)."""
    sents = [s.strip() for s in sent_tokenize(text) if len(s.strip()) > 20]
    return [" ".join(sents[i:i + n]) for i in range(0, len(sents), n)]

# =========================================================
# LOAD InLegalBERT  (embeddings for MMR + RAG retriever)
# =========================================================

print("\n📚 Loading InLegalBERT for MMR & RAG retrieval...")

legal_tokenizer_bert = AutoTokenizer.from_pretrained("law-ai/InLegalBERT")
legal_model_bert     = AutoModel.from_pretrained("law-ai/InLegalBERT").to(device)
legal_model_bert.eval()

# SentenceTransformer wrapper used for fast RAG chunk retrieval
retriever = SentenceTransformer(RETRIEVER_MODEL, device=device)

print("✅ InLegalBERT loaded (MMR + RAG retriever)")


@torch.no_grad()
def embed_with_legalbert(texts, batch_size=16):
    """Mean-pooled InLegalBERT embeddings (used for MMR sentence selection)."""
    if not texts:
        return np.array([])
    embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = legal_tokenizer_bert(
            batch, padding=True, truncation=True,
            max_length=512, return_tensors="pt"
        ).to(device)
        out  = legal_model_bert(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (out * mask).sum(1) / mask.sum(1)
        embs.append(pooled.cpu().numpy())
    return np.vstack(embs)

# =========================================================
# MMR SENTENCE SELECTION  (from Script 2)
# =========================================================

def select_sentences_mmr(judgment, k=60, lambda_param=MMR_LAMBDA):
    sentences   = sent_tokenize(judgment)
    n_original  = len(sentences)

    if len(sentences) <= k:
        return judgment, sentences, {
            "method": "no_filtering",
            "original_sents": n_original,
            "selected_sents": n_original
        }

    embeddings = embed_with_legalbert(sentences)
    centroid   = embeddings.mean(axis=0, keepdims=True)
    relevance  = cosine_similarity(embeddings, centroid).squeeze()

    selected_indices = [int(np.argmax(relevance))]
    remaining        = set(range(len(sentences))) - set(selected_indices)

    while len(selected_indices) < k and remaining:
        mmr_scores = []
        for i in remaining:
            sim_to_sel = cosine_similarity(
                embeddings[i:i+1], embeddings[selected_indices]
            ).max()
            mmr_scores.append(
                (i, lambda_param * relevance[i] - (1 - lambda_param) * sim_to_sel)
            )
        best = max(mmr_scores, key=lambda x: x[1])[0]
        selected_indices.append(best)
        remaining.remove(best)

    selected_indices.sort()
    selected_sentences = [sentences[i] for i in selected_indices]
    filtered_text      = " ".join(selected_sentences)

    return filtered_text, selected_sentences, {
        "method": "mmr",
        "original_sents": n_original,
        "selected_sents": len(selected_indices),
        "compression_ratio": len(selected_indices) / n_original,
        "lambda": lambda_param
    }

print("✅ MMR sentence selection ready")

# =========================================================
# LOAD GENERATION MODELS  (LED + Legal-Pegasus)
# =========================================================

print("\n📚 Loading generation models...")

models     = {}
tokenizers = {}

# ── LED ──────────────────────────────────────────────────
print("\n  [1/2] Loading LED...")
tokenizers["LED"] = LEDTokenizer.from_pretrained(MODEL_PATHS["LED"])
models["LED"]     = LEDForConditionalGeneration.from_pretrained(MODEL_PATHS["LED"]).to(device)
models["LED"].eval()
print(f"  ✅ LED loaded — {models['LED'].num_parameters():,} parameters")

# ── Legal-Pegasus ─────────────────────────────────────────
print("\n  [2/2] Loading Legal-Pegasus...")
tokenizers["PEGASUS"] = PegasusTokenizer.from_pretrained(MODEL_PATHS["PEGASUS"])
models["PEGASUS"]     = PegasusForConditionalGeneration.from_pretrained(MODEL_PATHS["PEGASUS"]).to(device)
models["PEGASUS"].eval()
print(f"  ✅ Legal-Pegasus loaded — {models['PEGASUS'].num_parameters():,} parameters")

print("\n✅ Both models loaded!")

# =========================================================
# RAG RETRIEVAL HELPERS  (from Script 1)
# =========================================================

def build_chunk_corpus(text, method=RETRIEVE_BY):
    """Build retrieval corpus from judgment text."""
    if method == "sentence":
        return chunk_into_sentences(text)
    else:
        # Use LED tokenizer for stable tokenisation
        return chunk_by_tokens(text, tokenizers["LED"],
                                max_tokens=CHUNK_SIZE_TOKENS,
                                overlap=CHUNK_OVERLAP_TOKENS)

def retrieve_top_k(chunks, query_embedding, k=TOP_K):
    """Retrieve k most relevant chunks using cosine similarity."""
    chunk_embs = retriever.encode(chunks, convert_to_tensor=True, show_progress_bar=False)
    scores     = util.cos_sim(query_embedding, chunk_embs)[0]
    topk_idx   = torch.topk(scores, min(k, len(chunks)))[1].cpu().numpy().tolist()
    return [(i, chunks[i], float(scores[i].cpu().item())) for i in topk_idx]

# =========================================================
# RAG-BASED CONTEXT BUILDER  (Script 1 workflow)
# =========================================================

def build_rag_context(judgment_text, current_k=TOP_K):
    """
    Retrieve top-k relevant chunks from the judgment using InLegalBERT
    and return a prompt-ready context string.
    """
    chunks = build_chunk_corpus(judgment_text)
    if not chunks:
        return ""
    query_emb = retriever.encode(judgment_text, convert_to_tensor=True, show_progress_bar=False)
    retrieved  = retrieve_top_k(chunks, query_emb, k=current_k)
    return " ".join([r[1] for r in retrieved])

# =========================================================
# GENERATION WITH RAG + LENGTH CONTROL  (Scripts 1 & 2 merged)
# =========================================================

PROMPT = ("Summarize the following legal judgment excerpts into a coherent "
          "abstractive summary (400-500 words):\n\n")

def generate_summary_rag(model_name, judgment_text):
    """
    Generate a summary for one judgment using:
      1. MMR sentence selection (Script 2)
      2. RAG chunk retrieval     (Script 1)
      3. Model-specific generation with length control (both scripts)
    Returns: (summary_string, selection_info_dict)
    """
    model     = models[model_name]
    tokenizer = tokenizers[model_name]
    config    = MODEL_CONFIG[model_name]

    # ── Step 1: MMR pre-filtering ────────────────────────
    k_mmr = MMR_K_LED if model_name == "LED" else MMR_K
    mmr_text, _, selection_info = select_sentences_mmr(judgment_text, k=k_mmr)

    # ── Step 2: RAG retrieval from original full text ────
    current_k     = TOP_K
    rag_context   = build_rag_context(judgment_text, current_k=current_k)
    gen_input     = PROMPT + rag_context

    # ── Step 3: Generate ─────────────────────────────────
    def _generate(input_text):
        if model_name == "LED":
            inputs = tokenizer(
                input_text, return_tensors="pt",
                truncation=True, max_length=config["max_input"]
            ).to(device)
            global_attn              = torch.zeros_like(inputs["attention_mask"])
            global_attn[:, 0]        = 1
            with torch.no_grad():
                ids = model.generate(
                    inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    global_attention_mask=global_attn,
                    max_length=GEN_MAX_TOKENS,
                    min_length=GEN_MIN_TOKENS,
                    num_beams=config["num_beams"],
                    length_penalty=1.0,
                    no_repeat_ngram_size=3,
                    early_stopping=True
                )
        else:  # PEGASUS
            inputs = tokenizer(
                input_text, return_tensors="pt",
                truncation=True, max_length=config["max_input"]
            ).to(device)
            with torch.no_grad():
                ids = model.generate(
                    inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    max_length=GEN_MAX_TOKENS,
                    min_length=GEN_MIN_TOKENS,
                    num_beams=config["num_beams"],
                    length_penalty=1.0,
                    no_repeat_ngram_size=3,
                    early_stopping=True
                )
        return tokenizer.decode(ids[0], skip_special_tokens=True,
                                clean_up_tokenization_spaces=True)

    summary = _generate(gen_input)

    # ── Step 4: RAG expansion loop (Script 1 logic) ──────
    all_chunks = build_chunk_corpus(judgment_text)
    round_idx  = 1
    while (words_count(summary) < MIN_WORDS or words_count(summary) > MAX_WORDS) \
            and round_idx <= MAX_RETRIEVE_ROUNDS:
        current_k  = min(len(all_chunks), current_k + TOP_K)
        rag_context = build_rag_context(judgment_text, current_k=current_k)

        # For long expansions prefer LED (handles more context); PEGASUS falls back to MMR text
        if model_name == "PEGASUS" and words_count(rag_context) > 600:
            gen_input = PROMPT + mmr_text   # fall back to MMR-filtered text
        else:
            gen_input = PROMPT + rag_context

        summary   = _generate(gen_input)
        round_idx += 1

    # ── Step 5: Post-process to enforce word limits ──────
    final_summary = truncate_to_word_limit_by_sentences(summary)
    return final_summary, selection_info

# =========================================================
# LOAD TEST DATA
# =========================================================

print(f"\n📂 Loading test data from {TEST_JSON_PATH}...")
with open(TEST_JSON_PATH, 'r', encoding='utf8') as f:
    test_data = json.load(f)
print(f"✅ Loaded {len(test_data)} test samples")

# =========================================================
# EVALUATION METRICS
# =========================================================

print("\n📊 Loading evaluation metrics...")
rouge_metric    = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")
print("✅ Metrics loaded: ROUGE, BERTScore")


def compute_metrics(predictions, references):
    rouge_scores = rouge_metric.compute(
        predictions=predictions, references=references, use_stemmer=True
    )
    bert_scores = bertscore_metric.compute(
        predictions=predictions, references=references,
        model_type="roberta-large", lang="en",
        device=device, batch_size=8
    )
    return {
        "rouge1":              float(rouge_scores["rouge1"]),
        "rouge2":              float(rouge_scores["rouge2"]),
        "rougeL":              float(rouge_scores["rougeL"]),
        "bertscore_precision": float(np.mean(bert_scores["precision"])),
        "bertscore_recall":    float(np.mean(bert_scores["recall"])),
        "bertscore_f1":        float(np.mean(bert_scores["f1"])),
    }

# =========================================================
# ENSEMBLE FUSION STRATEGIES  (from Script 2)
# =========================================================

def ensemble_voting(summaries_dict):
    all_sents = []
    for summary in summaries_dict.values():
        all_sents.extend(sent_tokenize(summary))
    counts   = Counter(s.lower().strip() for s in all_sents)
    selected = [s for s, c in counts.items() if c == 2]
    if len(selected) < 5:
        selected = [s for s, _ in counts.most_common(15)]
    return " ".join(selected)

def ensemble_weighted_concat(summaries_dict, weights):
    parts = []
    for model_name, summary in summaries_dict.items():
        sents  = sent_tokenize(summary)
        n      = max(1, int(len(sents) * weights[model_name]))
        parts.extend(sents[:n])
    seen, unique = set(), []
    for s in parts:
        key = s.lower().strip()
        if key not in seen:
            seen.add(key)
            unique.append(s)
    return " ".join(unique)

def ensemble_best_model(summaries_dict, reference):
    best_score, best_summary = -1, ""
    for summary in summaries_dict.values():
        score = rouge_metric.compute(
            predictions=[summary], references=[reference], use_stemmer=True
        )["rouge2"]
        if score > best_score:
            best_score, best_summary = score, summary
    return best_summary

def ensemble_sentence_ranking(summaries_dict):
    positions = {}
    for summary in summaries_dict.values():
        for pos, sent in enumerate(sent_tokenize(summary)):
            key = sent.lower().strip()
            positions.setdefault(key, []).append(pos)
    ranked = sorted(positions.items(), key=lambda x: np.mean(x[1]))
    return " ".join(s for s, _ in ranked[:15])

# =========================================================
# MAIN GENERATION LOOP
# =========================================================

print("\n" + "="*70)
print("🚀 GENERATING SUMMARIES — LED + Legal-Pegasus  (RAG + MMR)")
print("="*70)

all_model_summaries = {"LED": [], "PEGASUS": []}
ensemble_summaries  = {"voting": [], "weighted": [], "best_model": [], "ranking": []}
references          = []

for item in tqdm(test_data, desc="Processing samples"):
    judgment  = clean_judgment_text(item["judgment"])
    reference = item["reference_summary"]
    references.append(reference)

    summaries_dict = {}
    for model_name in ["LED", "PEGASUS"]:
        summary, _ = generate_summary_rag(model_name, judgment)
        all_model_summaries[model_name].append(summary)
        summaries_dict[model_name] = summary

    ensemble_summaries["voting"].append(ensemble_voting(summaries_dict))
    ensemble_summaries["weighted"].append(ensemble_weighted_concat(summaries_dict, ENSEMBLE_WEIGHTS))
    ensemble_summaries["best_model"].append(ensemble_best_model(summaries_dict, reference))
    ensemble_summaries["ranking"].append(ensemble_sentence_ranking(summaries_dict))

print("\n✅ All summaries generated!")

# =========================================================
# EVALUATION
# =========================================================

print("\n" + "="*70)
print("📊 EVALUATING ALL APPROACHES")
print("="*70)

results = {}

for model_name in ["LED", "PEGASUS"]:
    print(f"\n  Evaluating {model_name}...")
    metrics = compute_metrics(all_model_summaries[model_name], references)
    results[model_name] = metrics
    print(f"  ✅ {model_name} — ROUGE-2: {metrics['rouge2']:.4f}  BERTScore F1: {metrics['bertscore_f1']:.4f}")

for strategy in ["voting", "weighted", "best_model", "ranking"]:
    print(f"\n  Evaluating Ensemble-{strategy}...")
    metrics = compute_metrics(ensemble_summaries[strategy], references)
    results[f"ensemble_{strategy}"] = metrics
    print(f"  ✅ Ensemble-{strategy} — ROUGE-2: {metrics['rouge2']:.4f}  BERTScore F1: {metrics['bertscore_f1']:.4f}")

# =========================================================
# RESULTS TABLE
# =========================================================

print("\n" + "="*70)
print("📊 COMPREHENSIVE RESULTS COMPARISON")
print("="*70)
print(f"\n{'Approach':<25} {'ROUGE-1':<10} {'ROUGE-2':<10} {'ROUGE-L':<10} {'BERTScore F1':<12}")
print("-" * 70)
for name, m in results.items():
    print(f"{name:<25} {m['rouge1']:<10.4f} {m['rouge2']:<10.4f} {m['rougeL']:<10.4f} {m['bertscore_f1']:<12.4f}")

best_approach = max(results.items(), key=lambda x: x[1]['rouge2'])
print("\n" + "="*70)
print(f"🏆 BEST APPROACH: {best_approach[0].upper()}")
print(f"   ROUGE-2:      {best_approach[1]['rouge2']:.4f}")
print(f"   BERTScore F1: {best_approach[1]['bertscore_f1']:.4f}")
print("="*70)

# =========================================================
# SAVE RESULTS
# =========================================================

print("\n💾 Saving results...")

for model_name in ["LED", "PEGASUS"]:
    path = os.path.join(OUTPUT_DIR, f"{model_name.lower()}_summaries.json")
    with open(path, 'w', encoding='utf8') as f:
        json.dump([
            {"id": item.get("id", idx), "generated_summary": summary, "reference_summary": ref}
            for idx, (item, summary, ref) in enumerate(
                zip(test_data, all_model_summaries[model_name], references))
        ], f, indent=2, ensure_ascii=False)
    print(f"  ✅ {path}")

for strategy in ["voting", "weighted", "best_model", "ranking"]:
    path = os.path.join(OUTPUT_DIR, f"ensemble_{strategy}_summaries.json")
    with open(path, 'w', encoding='utf8') as f:
        json.dump([
            {"id": item.get("id", idx), "generated_summary": summary, "reference_summary": ref}
            for idx, (item, summary, ref) in enumerate(
                zip(test_data, ensemble_summaries[strategy], references))
        ], f, indent=2, ensure_ascii=False)
    print(f"  ✅ {path}")

complete = {
    "experiment":       "Ensemble RAG Summarization — LED + Legal-Pegasus",
    "test_samples":     len(test_data),
    "rag_config": {
        "retriever":         RETRIEVER_MODEL,
        "chunk_size_tokens": CHUNK_SIZE_TOKENS,
        "overlap_tokens":    CHUNK_OVERLAP_TOKENS,
        "top_k":             TOP_K,
        "max_rounds":        MAX_RETRIEVE_ROUNDS,
        "retrieve_by":       RETRIEVE_BY
    },
    "generation_config": {
        "min_words": MIN_WORDS,
        "max_words": MAX_WORDS,
        "min_tokens": GEN_MIN_TOKENS,
        "max_tokens": GEN_MAX_TOKENS
    },
    "ensemble_weights": ENSEMBLE_WEIGHTS,
    "results":          results,
    "best_approach": {"name": best_approach[0], "metrics": best_approach[1]}
}

results_path = os.path.join(OUTPUT_DIR, "ensemble_rag_complete_results.json")
with open(results_path, 'w', encoding='utf8') as f:
    json.dump(complete, f, indent=2, ensure_ascii=False)
print(f"  ✅ {results_path}")

# =========================================================
# REPORT
# =========================================================

report = [
    "="*70,
    "ENSEMBLE RAG SUMMARIZATION EXPERIMENT",
    "Models: LED (fine-tuned) + nsi319/legal-pegasus",
    "="*70,
    "",
    f"Test Samples:     {len(test_data)}",
    f"RAG Retriever:    {RETRIEVER_MODEL}",
    f"Chunk Size:       {CHUNK_SIZE_TOKENS} tokens  (overlap {CHUNK_OVERLAP_TOKENS})",
    f"Top-K Chunks:     {TOP_K}  (max expansion rounds: {MAX_RETRIEVE_ROUNDS})",
    f"MMR Config:       k={MMR_K} (PEGASUS)  k={MMR_K_LED} (LED)  λ={MMR_LAMBDA}",
    f"Target Length:    {MIN_WORDS}–{MAX_WORDS} words",
    "",
    "-"*70, "INDIVIDUAL MODEL RESULTS", "-"*70,
]
for model_name in ["LED", "PEGASUS"]:
    m = results[model_name]
    report += [
        f"\n{model_name}:",
        f"  ROUGE-1:      {m['rouge1']:.4f}",
        f"  ROUGE-2:      {m['rouge2']:.4f}",
        f"  ROUGE-L:      {m['rougeL']:.4f}",
        f"  BERTScore F1: {m['bertscore_f1']:.4f}",
    ]
report += ["", "-"*70, "ENSEMBLE RESULTS", "-"*70]
for strategy in ["voting", "weighted", "best_model", "ranking"]:
    m = results[f"ensemble_{strategy}"]
    report += [
        f"\nEnsemble-{strategy.upper()}:",
        f"  ROUGE-1:      {m['rouge1']:.4f}",
        f"  ROUGE-2:      {m['rouge2']:.4f}",
        f"  ROUGE-L:      {m['rougeL']:.4f}",
        f"  BERTScore F1: {m['bertscore_f1']:.4f}",
    ]
report += [
    "", "="*70,
    f"🏆 BEST: {best_approach[0].upper()}",
    f"   ROUGE-2:      {best_approach[1]['rouge2']:.4f}",
    f"   BERTScore F1: {best_approach[1]['bertscore_f1']:.4f}",
    "="*70,
]

report_text = "\n".join(report)
print("\n" + report_text)

report_path = os.path.join(OUTPUT_DIR, "ensemble_rag_report.txt")
with open(report_path, 'w', encoding='utf8') as f:
    f.write(report_text)
print(f"\n💾 Report saved: {report_path}")

print("\n" + "="*70)
print("✅ ENSEMBLE RAG EXPERIMENT COMPLETE")
print("="*70)
print(f"\nModels:     LED  +  nsi319/legal-pegasus")
print(f"Approaches: 2 individual  +  4 ensemble strategies")
print(f"Best:       {best_approach[0]}")
print("="*70)

🚀 Device: cuda
📂 Output directory: ./ensemble_rag_results
📂 Test data:        test.json

📚 Loading InLegalBERT for MMR & RAG retrieval...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
No sentence-transformers model found with name law-ai/InLegalBERT. Creating a new one wi

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...


Processing samples:   3%|█▊                                                          | 3/100 [09:28<5:05:59, 189.27s/it]Input ids are automatically padded from 3528 to 4096 to be a multiple of `config.attention_window`: 1024

Processing samples:   4%|██▍                                                         | 4/100 [12:21<4:52:50, 183.02s/it]Input ids are automatically padded from 3120 to 4096 to be a multiple of `config.attention_window`: 1024

Processing samples:   9%|█████▍                                                      | 9/100 [27:42<4:37:47, 183.16s/it]Input ids are automatically padded from 2251 to 3072 to be a multiple of `config.attention_window`: 1024

Processing samples:  10%|█████▉                                                     | 10/100 [29:46<4:07:23, 164.93s/it]Input ids are automatically padded from 3044 to 3072 to be a multiple of `config.attention_window`: 1024

Processing samples:  17%|██████████                                                 | 17/100 [5


✅ All summaries generated!

📊 EVALUATING ALL APPROACHES

  Evaluating LED...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ✅ LED — ROUGE-2: 0.2846  BERTScore F1: 0.8527

  Evaluating PEGASUS...
  ✅ PEGASUS — ROUGE-2: 0.0001  BERTScore F1: 0.7092

  Evaluating Ensemble-voting...
  ✅ Ensemble-voting — ROUGE-2: 0.2406  BERTScore F1: 0.8456

  Evaluating Ensemble-weighted...
  ✅ Ensemble-weighted — ROUGE-2: 0.1252  BERTScore F1: 0.8049

  Evaluating Ensemble-best_model...
  ✅ Ensemble-best_model — ROUGE-2: 0.2846  BERTScore F1: 0.8527

  Evaluating Ensemble-ranking...
  ✅ Ensemble-ranking — ROUGE-2: 0.1639  BERTScore F1: 0.7312

📊 COMPREHENSIVE RESULTS COMPARISON

Approach                  ROUGE-1    ROUGE-2    ROUGE-L    BERTScore F1
----------------------------------------------------------------------
LED                       0.5203     0.2846     0.2819     0.8527      
PEGASUS                   0.0421     0.0001     0.0238     0.7092      
ensemble_voting           0.4496     0.2406     0.2491     0.8456      
ensemble_weighted         0.2661     0.1252     0.1485     0.8049      
ensemble_best_model  